# 18｜Transformer_注意力架構與預訓練

位置編碼加上 causal mask，驗證第 t 個 token 不會看到未來。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/18_Transformer_注意力架構與預訓練.md`，目前 32 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

完整小型 attention block、兩種 mask、MLM/任務頭/QA 與蒸餾。

保留教材原圖；同概念重畫不保證原書分數相同。架構圖以機制或形狀對照，完整模型案例的範圍在各格說明。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 18-03 | 6、8、15、16 | 課堂小實驗 |
| 18-04 | 3、15、16 | 執行結果圖 |
| 18-05 | 4、5、6、7、8、9、10、11、12、13、14、15、16、17、18、19、20、24、29、30、32 | 1. 多頭 attention、位置與完整 block 前向 |
| 18-06 | 21、22、23、24、25、26、32 | 2. MLM、句子分類、token 分類與問答頭 |
| 18-07 | 27、28、31、32 | 3. Teacher–student 的軟標籤蒸餾 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 18-01｜投影片 18：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 18-02｜投影片 18：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 課堂小實驗

In [ ]:
# 程式 18-03｜投影片 18：6、8、15、16 頁
n, d = 6, 4
pos = np.arange(n)[:, None]
freq = np.exp(np.arange(0, d, 2) * (-np.log(10000.0) / d))
pe = np.zeros((n, d)); pe[:, 0::2] = np.sin(pos * freq); pe[:, 1::2] = np.cos(pos * freq)
scores = pe @ pe.T / np.sqrt(d)
causal_mask = np.triu(np.ones((n, n), dtype=bool), k=1)
masked = np.where(causal_mask, -1e9, scores)
weights = np.exp(masked - masked.max(axis=1, keepdims=True)); weights /= weights.sum(1, keepdims=True)
print('future attention mass:', weights[causal_mask].sum())
pd.DataFrame(weights).round(3)

## 執行結果圖

In [ ]:
# 程式 18-04｜投影片 18：3、15、16 頁
plt.imshow(weights, cmap='magma', vmin=0, vmax=1)
plt.colorbar(label='attention weight'); plt.xlabel('key position'); plt.ylabel('query position')
plt.title('Causal self-attention is lower triangular')
savefig('18_transformer_mask_demo')
report('transformer_mask', {'future_mass': weights[causal_mask].sum(), 'weights': weights.tolist()})

## 1. 多頭 attention、位置與完整 block 前向

用小尺寸 NumPy 實作 encoder self-attention、decoder causal self-attention、cross-attention、FFN 與殘差。權重隨機，僅驗證架構資料流，不稱為已預訓練 Transformer。

In [ ]:
# 程式 18-05｜投影片 18：4、5、6、7、8、9、10、11、12、13、14、15、16、17、18、19、20、24、29、30、32 頁
rng18=np.random.default_rng(SEED)
def softmax18(z):
    e=np.exp(z-z.max(axis=-1,keepdims=True));return e/e.sum(axis=-1,keepdims=True)
def norm18(x):return (x-x.mean(-1,keepdims=True))/np.sqrt(x.var(-1,keepdims=True)+1e-5)
batch18,length18,width18,heads18=2,5,8,2
x18=rng18.normal(size=(batch18,length18,width18));source18=rng18.normal(size=(batch18,4,width18))
Wq18,Wk18,Wv18,Wo18=[rng18.normal(0,.2,(width18,width18)) for _ in range(4)]
def multi18(query,source,allowed=None):
    B,L,D=query.shape;S=source.shape[1];dh=D//heads18
    q=(query@Wq18).reshape(B,L,heads18,dh).transpose(0,2,1,3)
    k=(source@Wk18).reshape(B,S,heads18,dh).transpose(0,2,1,3)
    v=(source@Wv18).reshape(B,S,heads18,dh).transpose(0,2,1,3)
    score=q@k.transpose(0,1,3,2)/np.sqrt(dh)
    if allowed is not None:score=np.where(allowed[:,None],score,-np.inf)
    att=softmax18(score);out=(att@v).transpose(0,2,1,3).reshape(B,L,D)@Wo18
    return out,att
causal18=np.broadcast_to(np.tril(np.ones((5,5),bool)),(2,5,5)).copy()
encoded18,_=multi18(source18,source18);encoded18=norm18(source18+encoded18)
self18,att18=multi18(x18,x18,causal18);decoder18=norm18(x18+self18)
valid_source18=np.array([[1,1,1,1],[1,1,0,0]],bool)
allowed_cross18=np.broadcast_to(valid_source18[:,None,:],(2,5,4))
cross18,cross_weights18=multi18(decoder18,encoded18,allowed_cross18)
decoder18=norm18(decoder18+cross18)
W118=rng18.normal(0,.2,(8,16));W218=rng18.normal(0,.2,(16,8))
block18=norm18(decoder18+np.maximum(decoder18@W118,0)@W218)
assert block18.shape==x18.shape and np.all(cross_weights18[1,:,:,2:]==0)
# 改未來 token，前兩步 causal self-attention 不變
changed18=x18.copy();changed18[:,2:]+=10;changed_out18,_=multi18(changed18,changed18,causal18)
assert np.allclose(self18[:,:2],changed_out18[:,:2])
fig,axes=plt.subplots(1,3,figsize=(11,3))
for ax,a,title in zip(axes,[att18[0,0],att18[0,1],cross_weights18[1,0]],['causal head 1','causal head 2','cross, masked source']):ax.imshow(a,cmap='magma',vmin=0,vmax=1);ax.set_title(title);ax.set(xlabel='key',ylabel='query')
savefig('c18_multihead_masks')
# 手算注意力與 LayerNorm
hand18=softmax18(np.array([1/np.sqrt(2),0]));print('Hand weights/context:',hand18,hand18@np.array([[2,0],[0,4]]))
ln18=np.array([2.,4.,4.,6.]);assert np.allclose(norm18(ln18),[-np.sqrt(2),0,0,np.sqrt(2)],atol=1e-5)
# pre-norm 與 post-norm 是不同函數
f18=lambda z:np.maximum(z@W118,0)@W218
assert not np.allclose(x18+f18(norm18(x18)),norm18(x18+f18(x18)))
learned_positions18=rng18.normal(0,.02,(5,8));print('Learned position table / embedded shape:',learned_positions18.shape,(x18+learned_positions18).shape)
plt.imshow(pe,aspect='auto',cmap='coolwarm');plt.xlabel('channel');plt.ylabel('position');plt.title('Fixed sinusoidal position encoding');savefig('c18_position_encoding')

## 2. MLM、句子分類、token 分類與問答頭

MLM 只在選中的位置算損失；被選中不代表一定換成 MASK。BERT 原始 NSP 用句對標籤。下例只有資料構造與損失，不假裝具備語意能力。

In [ ]:
# 程式 18-06｜投影片 18：21、22、23、24、25、26、32 頁
ids18=rng18.integers(3,20,size=(8,10));selected18=rng18.random(ids18.shape)<.15
choice18=rng18.random(ids18.shape);corrupted18=ids18.copy();corrupted18[selected18&(choice18<.8)]=1
random18=selected18&(choice18>=.8)&(choice18<.9);corrupted18[random18]=rng18.integers(3,20,size=random18.sum())
labels18=np.where(selected18,ids18,-100);vocab_logits18=rng18.normal(size=(8,10,20));logp18=np.log(softmax18(vocab_logits18))
mlm_loss18=-logp18[np.where(selected18)[0],np.where(selected18)[1],ids18[selected18]].mean()
assert np.all(labels18[~selected18]==-100)
print('Selected / masked / random / unchanged:',selected18.sum(),np.sum(selected18&(choice18<.8)),random18.sum(),np.sum(selected18&(choice18>=.9)))
print('MLM loss on selected positions:',mlm_loss18)
h18=rng18.normal(size=(2,6,8));cls18=h18[:,0]@rng18.normal(size=(8,3));tag18=h18@rng18.normal(size=(8,5));qa18=h18@rng18.normal(size=(8,2))
# QA 合法 span 須 start<=end，並排除 padding；此處每筆長度6
start18=softmax18(qa18[:,:,0]);end18=softmax18(qa18[:,:,1]);joint18=start18[:,:,None]*end18[:,None,:]
joint18=np.where(np.triu(np.ones((6,6),bool))[None],joint18,-1)
span18=[np.unravel_index(np.argmax(row),(6,6)) for row in joint18]
print('Classification / token / QA shapes:',cls18.shape,tag18.shape,qa18.shape,'spans:',span18)
fig,axes=plt.subplots(1,2,figsize=(10,3));axes[0].imshow(selected18,cmap='binary',aspect='auto');axes[0].set_title('MLM supervised positions')
axes[1].plot(start18[0],label='start');axes[1].plot(end18[0],label='end');axes[1].legend();axes[1].set_title('Question-answer span head');savefig('c18_mlm_task_heads')

## 3. Teacher–student 的軟標籤蒸餾

教師機率作為固定目標；學生更新 logits 以降低 KL。溫度 T 改變分布；乘 T² 調整梯度尺度。這是單一分布的損失實驗，不是 DistilBERT 訓練。

In [ ]:
# 程式 18-07｜投影片 18：27、28、31、32 頁
teacher18=np.array([2.,.5,-1.]);student18=np.zeros(3);T18=2.;target18=softmax18(teacher18/T18);trace18=[]
for _ in range(80):
    ps18=softmax18(student18/T18);kl18=T18*T18*np.sum(target18*np.log(target18/ps18));trace18.append(kl18)
    student18-=.2*T18*(ps18-target18)
assert trace18[-1]<trace18[0]
plt.plot(trace18);plt.xlabel('student update');plt.ylabel('T^2 KL(teacher || student)');savefig('c18_distillation_loss')
print('Teacher / student soft probabilities:',target18,softmax18(student18/T18))

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。